In [1]:
import pandas as pd # Use this to load and look at CSV file
from sklearn.model_selection import train_test_split # For splitting data into train vs test

In [2]:
df = pd.read_csv("../data/Tweets.csv") # Read the data file
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [3]:
df["airline_sentiment"].value_counts()

airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

Text preprocessing

In [4]:
import re
import string
import nltk
from nltk.corpus import stopwords

# Download stopword list on first run (safe to re-run — it skips if already present)
nltk.download("stopwords", quiet=True)

STOPWORDS = set(stopwords.words("english"))
print(f"Loaded {len(STOPWORDS)} English stopwords")

Loaded 198 English stopwords


In [5]:
# Precompile regex patterns once (faster than recompiling for every tweet)
URL_PATTERN = re.compile(r"http\S+|www\.\S+")      # matches http://..., https://..., www...
MENTION_PATTERN = re.compile(r"@(\w+)")             # matches @username, captures just the name
HASHTAG_PATTERN = re.compile(r"#(\w+)")             # matches #word, captures just the word
NON_ASCII_PATTERN = re.compile(r"[^\x00-\x7F]+")    # matches anything outside basic ASCII (emojis, etc.)
NUMBER_PATTERN = re.compile(r"\d+")                 # matches one or more digits


def preprocess_tweet(text: str) -> str:
    
    "lowercase --> no urls --> no # --> no emojis --> no numbers --> no punctuation --> create tokens"

    # 1. Lowercase
    text = text.lower()

    # 2. Remove URLs
    text = URL_PATTERN.sub(" ", text)

    # 3. Strip @ but keep the mention word (airline name is useful signal)
    text = MENTION_PATTERN.sub(r"\1", text)

    # 4. Strip # but keep the hashtag word
    text = HASHTAG_PATTERN.sub(r"\1", text)

    # 5. Remove emojis / non-ASCII
    text = NON_ASCII_PATTERN.sub(" ", text)

    # 6. Remove numbers
    text = NUMBER_PATTERN.sub(" ", text)

    # 7. Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))

    # 8. Tokenize + filter stopwords + drop single-char tokens
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 1]

    # 9. Rejoin
    return " ".join(tokens)


# Quick test
sample = "@VirginAmerica what @dhepburn said. Really excited for my flight! 😊 http://t.co/abc123 #flying"
print("BEFORE:", sample)
print("AFTER: ", preprocess_tweet(sample))

BEFORE: @VirginAmerica what @dhepburn said. Really excited for my flight! 😊 http://t.co/abc123 #flying
AFTER:  virginamerica dhepburn said really excited flight flying


In [6]:
# Apply preprocessing to every tweet, create a new clean_text column
df["clean_text"] = df["text"].apply(preprocess_tweet)

# Confirm it worked
print(f"Processed {len(df)} tweets")
print(f"Columns now: {list(df.columns)}")
df[["text", "clean_text"]].head()

Processed 14640 tweets
Columns now: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone', 'clean_text']


,text,clean_text
0,@VirginAmerica What @dhepburn said.,virginamerica dhepburn said
1,@VirginAmerica plus you've added commercials t...,virginamerica plus youve added commercials exp...
2,@VirginAmerica I didn't today... Must mean I n...,virginamerica didnt today must mean need take ...
3,@VirginAmerica it's really aggressive to blast...,virginamerica really aggressive blast obnoxiou...
4,@VirginAmerica and it's a really big bad thing...,virginamerica really big bad thing


In [7]:
# Show before/after examples, one from each sentiment class
for sentiment in ["positive", "negative", "neutral"]:
    sample = df[df["airline_sentiment"] == sentiment].sample(2, random_state=42)
    print(f"\n{'='*70}")
    print(f"  {sentiment.upper()} TWEETS")
    print('='*70)
    for _, row in sample.iterrows():
        print(f"\nBEFORE: {row['text']}")
        print(f"AFTER : {row['clean_text']}")


  POSITIVE TWEETS

BEFORE: @SouthwestAir thanks for your excellent response time and assistance! All set :)
AFTER : southwestair thanks excellent response time assistance set

BEFORE: @JetBlue thanks. I appreciate your prompt response.
AFTER : jetblue thanks appreciate prompt response

  NEGATIVE TWEETS

BEFORE: @united gate C 24 IAD. U released passengers to board w/others deplaning .50 peopleOn bridge while next flight  board http://t.co/HfoF33iyhi
AFTER : united gate iad released passengers board wothers deplaning peopleon bridge next flight board

BEFORE: @USAirways 1729 connecting in charlotte to houston. Mechanical issue determined while q'd to take off. And we checked our bags.
AFTER : usairways connecting charlotte houston mechanical issue determined qd take checked bags

  NEUTRAL TWEETS

BEFORE: @united we finally just arrive to Bogota, good but long flight!!
AFTER : united finally arrive bogota good long flight

BEFORE: @AmericanAir got a callback at 1 am, took care of it. 

In [8]:
X = df["clean_text"]
y = df["airline_sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

In [9]:
print(X_train.shape, X_test.shape)

(11712,) (2928,)


Feature Extraction

In [10]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# --- Bag of Words ---
bow_vectorizer = CountVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
)
X_train_bow = bow_vectorizer.fit_transform(X_train)   # fit on train
X_test_bow  = bow_vectorizer.transform(X_test)         # transform test with train's vocab

# --- TF-IDF ---
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf  = tfidf_vectorizer.transform(X_test)

print("Vectorization complete.")

Vectorization complete.


In [11]:
print(f"BoW:   training matrix has shape {X_train_bow.shape}")
print(f"       test matrix has shape     {X_test_bow.shape}")
print(f"TF-IDF: training matrix has shape {X_train_tfidf.shape}")
print(f"        test matrix has shape     {X_test_tfidf.shape}")

# Peek at a few vocabulary items to confirm what we got
vocab = bow_vectorizer.get_feature_names_out()
print(f"\nTotal unique features in BoW: {len(vocab)}")
print(f"Sample features (every 500th): {vocab[::500].tolist()}")

BoW:   training matrix has shape (11712, 5000)
       test matrix has shape     (2928, 5000)
TF-IDF: training matrix has shape (11712, 5000)
        test matrix has shape     (2928, 5000)

Total unique features in BoW: 5000
Sample features (every 500th): ['aa', 'bhm', 'degrees', 'flight delay', 'heard anything', 'lesson', 'one', 'rescheduling', 'supposed', 'united ua']


## Naive Bayes

In [12]:
from sklearn.naive_bayes import MultinomialNB

# Step 1: Use Bag-of-Words (BoW) to count every word in each message
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
y_predict_nb_bow = nb_bow.predict(X_test_bow)

# Next Train Naive Bayes on term frequency and if they are infomative or not
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
y_predict_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

print("Naive Bayes models are now trained on both the Bag of Words (BoW) and Term Frequency + Inverse Document Frequency (TF-IDF)")

Naive Bayes models are now trained on both the Bag of Words (BoW) and Term Frequency + Inverse Document Frequency (TF-IDF)


## Logistic Regression

In [13]:
from sklearn.linear_model import LogisticRegression

# Logistic Regression using Bag-of-Words features
lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)
y_predict_lr_bow = lr_bow.predict(X_test_bow)

# Logistic Regression using TF-IDF features
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)
y_predict_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

print("Logistic Regression models are now trained on both BoW and TF-IDF")

Logistic Regression models are now trained on both BoW and TF-IDF


## Model Testing, How correct each model is 

In [14]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    "Naive Bayes + BoW": y_predict_nb_bow,
    "Naive Bayes + TF-IDF": y_predict_nb_tfidf,
    "Logistic Regression + BoW": y_predict_lr_bow,
    "Logistic Regression + TF-IDF": y_predict_lr_tfidf
}

for model_name, predictions in models.items():
    print("=" * 70)
    print(model_name)
    print("=" * 70)

    print("Accuracy:", accuracy_score(y_test, predictions))

    print("\nClassification Report:")
    print(classification_report(y_test, predictions))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, predictions))

    print("\n")

Naive Bayes + BoW
Accuracy: 0.7718579234972678

Classification Report:
              precision    recall  f1-score   support

    negative       0.83      0.88      0.86      1835
     neutral       0.63      0.51      0.56       620
    positive       0.68      0.67      0.68       473

    accuracy                           0.77      2928
   macro avg       0.71      0.69      0.70      2928
weighted avg       0.76      0.77      0.77      2928


Confusion Matrix:
[[1623  125   87]
 [ 237  318   65]
 [  89   65  319]]


Naive Bayes + TF-IDF
Accuracy: 0.7418032786885246

Classification Report:
              precision    recall  f1-score   support

    negative       0.74      0.98      0.84      1835
     neutral       0.73      0.28      0.41       620
    positive       0.82      0.43      0.56       473

    accuracy                           0.74      2928
   macro avg       0.76      0.56      0.60      2928
weighted avg       0.75      0.74      0.70      2928


Confusion Matrix

## Which model performed the best

In [15]:
results = []

for model_name, predictions in models.items():
    accuracy = accuracy_score(y_test, predictions)
    results.append([model_name, accuracy])

results_df = pd.DataFrame(results, columns=["Model", "Accuracy"])
results_df.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy
3,Logistic Regression + TF-IDF,0.791325
2,Logistic Regression + BoW,0.784153
0,Naive Bayes + BoW,0.771858
1,Naive Bayes + TF-IDF,0.741803


## Custom made test tweets

In [16]:
custom_tweet = ["My flight was delayed again and customer service was terrible"]

custom_tweet_clean = [preprocess_tweet(custom_tweet[0])]
custom_tweet_tfidf = tfidf_vectorizer.transform(custom_tweet_clean)

prediction = lr_tfidf.predict(custom_tweet_tfidf)
probabilities = lr_tfidf.predict_proba(custom_tweet_tfidf)

print("Cleaned tweet:", custom_tweet_clean[0])
print("Predicted sentiment:", prediction[0])
print("Probabilities:", probabilities)

Cleaned tweet: flight delayed customer service terrible
Predicted sentiment: negative
Probabilities: [[0.99286304 0.00225564 0.00488132]]


Final Conclusion:

Logistic Regression with TF-IDF achieved the highest accuracy among all models tested. 
This suggests that weighting words by importance (TF-IDF) combined with a more flexible model 
(Logistic Regression) provides better performance for sentiment classification.

Naive Bayes performed slightly worse, but not by much.

## Visualize Most Frequent Words

In [17]:
import numpy as np

# Get words and their counts from BoW
words = bow_vectorizer.get_feature_names_out()
counts = X_train_bow.toarray().sum(axis=0)

word_freq = dict(zip(words, counts))

top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:20]

print("Top 20 Most Frequent Words:")
for word, freq in top_words:
    print(word, freq)

Top 20 Most Frequent Words:
united 3302
flight 3116
usairways 2464
americanair 2394
southwestair 1937
jetblue 1878
get 1074
thanks 868
cancelled 856
service 735
help 695
time 613
im 596
customer 580
us 570
hours 561
flights 541
amp 515
hold 513
plane 498
